In [1]:
import pandas as pd
import numpy as np
test_data = pd.read_csv('/mnt/data/users/abdelrahman.sadallah/ONEDRIVE/poetry/test_data.csv')
train_data = pd.read_csv('/mnt/data/users/abdelrahman.sadallah/ONEDRIVE/poetry/georoge_decontaminated_train.csv')


# Convert poem_verses to numeric and remove rows with values < 2
initial_rows = len(train_data)
train_data['poem_verses'] = pd.to_numeric(train_data['poem_verses'], errors='coerce')
train_data = train_data[train_data['poem_verses'] >= 2].reset_index(drop=True)
print(f"Removed {initial_rows - len(train_data)} rows; remaining train_data rows: {len(train_data)}")

test_data['poem_verses'] = pd.to_numeric(test_data['poem_verses'], errors='coerce')
initial_test_rows = len(test_data)
test_data = test_data[test_data['poem_verses'] >= 2].reset_index(drop=True)
print(f"Removed {initial_test_rows - len(test_data)} rows; remaining test_data rows: {len(test_data)}")




/tmp/ipykernel_3976066/693717267.py:4: DtypeWarning: Columns (1,2,6,8,9,10,12,13,17,19,23,24,25,27,28) have mixed types. Specify dtype option on import or set low_memory=False.
  train_data = pd.read_csv('/mnt/data/users/abdelrahman.sadallah/ONEDRIVE/poetry/georoge_decontaminated_train.csv')


Removed 1877753 rows; remaining train_data rows: 427374
Removed 0 rows; remaining test_data rows: 6917


In [2]:
train_data['dataset_name'].value_counts()

dataset_name
boda_scrapped            220795
Ashaar                   123583
AraPoems                  62965
mawsooaa                  18004
arapoet                    1303
Arabic Poetry Dataset       662
Arabic-Poetry-Melody         48
poems_hakim                   8
adab_world                    6
Name: count, dtype: int64

In [4]:
def dataset_origin_stats(df, clean_col='poem_text_no_diacritics'):
    """
    Create statistics segmented by dataset origin.
    For 'boda_scrapped' dataset_name, further segment by source column.
    For other dataset_names, use dataset_name directly as origin.
    """
    tmp = df.copy()
    
    # Create an 'origin' column based on dataset_name and source
    def get_origin(row):
        dataset_name = str(row.get('dataset_name', 'UNKNOWN')).strip()
        if pd.isna(row.get('dataset_name')) or dataset_name == '' or dataset_name == 'nan':
            dataset_name = 'UNKNOWN'
        
        # If dataset_name is 'boda_scrapped', use source column
        if dataset_name.lower() == 'boda_scrapped':
            source = str(row.get('source', 'UNKNOWN')).strip()
            if pd.isna(row.get('source')) or source == '' or source == 'nan':
                source = 'UNKNOWN'
            return f"boda_scrapped/{source}"
        else:
            return dataset_name
    
    tmp['origin'] = tmp.apply(get_origin, axis=1)
    
    # Clean the text column
    tmp['clean'] = tmp[clean_col].astype(str).str.strip().replace('', pd.NA)
    tmp['char_len'] = tmp['clean'].str.len()
    tmp['verses'] = pd.to_numeric(tmp.get('poem_verses', pd.NA), errors='coerce')
    
    # Aggregate by origin
    agg = tmp.groupby('origin').agg(
        samples=('clean', 'size'),
        non_empty_samples=('char_len', lambda s: s.notna().sum()),
        avg_char_len=('char_len', 'mean'),
        avg_verses=('verses', 'mean')
    ).reset_index()
    
    agg[['avg_char_len', 'avg_verses']] = agg[['avg_char_len', 'avg_verses']].round(2)
    
    # Calculate totals for the summary row
    total_samples = agg['samples'].sum()
    total_non_empty_samples = agg['non_empty_samples'].sum()
    
    # Calculate weighted averages for character length and verses
    # Weight by non_empty_samples for avg_char_len and by samples for avg_verses
    weighted_avg_char_len = (agg['avg_char_len'] * agg['non_empty_samples']).sum() / total_non_empty_samples if total_non_empty_samples > 0 else 0
    weighted_avg_verses = (agg['avg_verses'] * agg['samples']).sum() / total_samples if total_samples > 0 else 0
    
    # Create total row
    total_row = pd.DataFrame({
        'origin': ['TOTAL'],
        'samples': [total_samples],
        'non_empty_samples': [total_non_empty_samples],
        'avg_char_len': [round(weighted_avg_char_len, 2)],
        'avg_verses': [round(weighted_avg_verses, 2)]
    })
    
    # Concatenate the aggregated data with the total row
    result = pd.concat([agg.sort_values('samples', ascending=False), total_row], ignore_index=True)
    
    return result

# Generate stats for train and test
train_origin_stats = dataset_origin_stats(train_data)
test_origin_stats = dataset_origin_stats(test_data)

# Add a 'split' column to identify train vs test
train_origin_stats['split'] = 'train'
test_origin_stats['split'] = 'test'

# Combine into one dataframe
combined_stats = pd.concat([train_origin_stats, test_origin_stats], ignore_index=True)

# Reorder columns for better readability
combined_stats = combined_stats[['split', 'origin', 'samples', 'non_empty_samples', 'avg_char_len', 'avg_verses']]

print("=== Train per-origin stats ===")
print(train_origin_stats.to_string(index=False))

print("\n=== Test per-origin stats ===")
print(test_origin_stats.to_string(index=False))

print("\n=== Combined stats (ready for LaTeX) ===")
print(combined_stats.to_string(index=False))

# Optional: Export to LaTeX format
print("\n=== LaTeX Table ===")
latex_table = combined_stats.to_latex(index=False, escape=False)
print(latex_table)

=== Train per-origin stats ===
                  origin  samples  non_empty_samples  avg_char_len  avg_verses split
                  Ashaar   123583             123583        907.63       19.81 train
boda_scrapped/poets_gate   112493             112493        681.52       15.58 train
      boda_scrapped/adab    70277              70277        905.53       35.33 train
                AraPoems    62965              62965       1039.49       22.01 train
     boda_scrapped/diwan    38025              38025        851.45       22.65 train
                mawsooaa    18004              18004        598.55       10.25 train
                 arapoet     1303               1303        536.40        9.25 train
   Arabic Poetry Dataset      662                662        970.59       19.41 train
    Arabic-Poetry-Melody       48                 48        938.79       21.44 train
             poems_hakim        8                  8       1053.62       24.88 train
              adab_world        6 

In [5]:
import re
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = re.sub(r"[\t\n\r]+", " ", text)
    text = re.sub(r"\s+", " ", text)
    text = text.strip()
    text = re.sub(r"ـ+", "", text)
    return text

# Clean poem_text_no_diacritics in both datasets
for df in (train_data, test_data):
    df['poem_text_no_diacritics_clean'] = df['poem_text_no_diacritics'].apply(clean_text)

# Helper to compute duplicate stats for a cleaned column
def duplicate_stats(df, col='poem_text_no_diacritics_clean'):
    total = len(df)
    non_empty_mask = df[col].str.strip() != ""
    non_empty = non_empty_mask.sum()
    unique_texts = df.loc[non_empty_mask, col].nunique()
    duplicate_rows = df.loc[non_empty_mask, col].duplicated(keep='first').sum()
    duplicate_values = non_empty - unique_texts
    top_dupes = df.loc[non_empty_mask, col].value_counts().head(10)
    return {
        "total_rows": total,
        "non_empty_rows": non_empty,
        "unique_texts": unique_texts,
        "duplicate_rows": duplicate_rows,
        "duplicate_values": duplicate_values,
        "top_duplicates": top_dupes
    }

# Compute and display duplicate stats for train and test datasets
print("=== DUPLICATES WITHIN EACH DATASET ===")
train_stats = duplicate_stats(train_data)
test_stats = duplicate_stats(test_data)

print(f"\nTRAIN DATA DUPLICATES:")
print(f"Total rows: {train_stats['total_rows']}")
print(f"Non-empty rows: {train_stats['non_empty_rows']}")
print(f"Unique texts: {train_stats['unique_texts']}")
print(f"Duplicate rows: {train_stats['duplicate_rows']}")

print(f"\nTEST DATA DUPLICATES:")
print(f"Total rows: {test_stats['total_rows']}")
print(f"Non-empty rows: {test_stats['non_empty_rows']}")
print(f"Unique texts: {test_stats['unique_texts']}")
print(f"Duplicate rows: {test_stats['duplicate_rows']}")

# Check for duplicates between train and test datasets
print("\n=== DUPLICATES BETWEEN TRAIN AND TEST DATASETS ===")

# Get non-empty clean texts from both datasets
train_clean_texts = set(train_data[train_data['poem_text_no_diacritics_clean'].str.strip() != '']['poem_text_no_diacritics_clean'])
test_clean_texts = set(test_data[test_data['poem_text_no_diacritics_clean'].str.strip() != '']['poem_text_no_diacritics_clean'])

# Find overlapping texts
overlapping_texts = train_clean_texts.intersection(test_clean_texts)

print(f"Unique non-empty texts in train: {len(train_clean_texts)}")
print(f"Unique non-empty texts in test: {len(test_clean_texts)}")
print(f"Overlapping texts between train and test: {len(overlapping_texts)}")

if len(overlapping_texts) > 0:
    print(f"\nOverlap percentage: {len(overlapping_texts) / len(train_clean_texts.union(test_clean_texts)) * 100:.2f}%")
    
else:
    print("No overlapping texts found between train and test datasets!")

print(f"\n=== SUMMARY ===")
print(f"Train duplicates within dataset: {train_stats['duplicate_rows']} rows")
print(f"Test duplicates within dataset: {test_stats['duplicate_rows']} rows") 
print(f"Cross-dataset duplicates: {len(overlapping_texts)} unique texts")

=== DUPLICATES WITHIN EACH DATASET ===

TRAIN DATA DUPLICATES:
Total rows: 2531845
Non-empty rows: 2531845
Unique texts: 2305127
Duplicate rows: 226718

TEST DATA DUPLICATES:
Total rows: 6917
Non-empty rows: 6917
Unique texts: 6917
Duplicate rows: 0

=== DUPLICATES BETWEEN TRAIN AND TEST DATASETS ===

TRAIN DATA DUPLICATES:
Total rows: 2531845
Non-empty rows: 2531845
Unique texts: 2305127
Duplicate rows: 226718

TEST DATA DUPLICATES:
Total rows: 6917
Non-empty rows: 6917
Unique texts: 6917
Duplicate rows: 0

=== DUPLICATES BETWEEN TRAIN AND TEST DATASETS ===
Unique non-empty texts in train: 2305127
Unique non-empty texts in test: 6917
Overlapping texts between train and test: 0
No overlapping texts found between train and test datasets!

=== SUMMARY ===
Train duplicates within dataset: 226718 rows
Test duplicates within dataset: 0 rows
Cross-dataset duplicates: 0 unique texts
Unique non-empty texts in train: 2305127
Unique non-empty texts in test: 6917
Overlapping texts between train a

In [ ]:
# Remove duplicates from train dataset and save back to original file
print("=== REMOVING DUPLICATES FROM TRAIN DATASET ===")

# Store original file path
original_train_file = '/mnt/data/users/abdelrahman.sadallah/ONEDRIVE/poetry/georoge_decontaminated_train.csv'

# Get initial counts
initial_count = len(train_data)
initial_non_empty = (train_data['poem_text_no_diacritics_clean'].str.strip() != "").sum()

print(f"Initial train dataset size: {initial_count} rows")
print(f"Initial non-empty texts: {initial_non_empty}")

# Remove duplicates based on cleaned text
# Keep the first occurrence of each duplicate
mask_non_empty = train_data['poem_text_no_diacritics_clean'].str.strip() != ""
train_data_dedup = train_data.copy()

# Remove rows with duplicate cleaned text (keeping first occurrence)
train_data_dedup = train_data_dedup.drop_duplicates(
    subset=['poem_text_no_diacritics_clean'], 
    keep='first'
).reset_index(drop=True)

# Get final counts
final_count = len(train_data_dedup)
final_non_empty = (train_data_dedup['poem_text_no_diacritics_clean'].str.strip() != "").sum()
removed_count = initial_count - final_count

print(f"\nAfter deduplication:")
print(f"Final train dataset size: {final_count} rows")
print(f"Final non-empty texts: {final_non_empty}")
print(f"Removed {removed_count} duplicate rows")

# Verify no duplicates remain
remaining_duplicates = train_data_dedup['poem_text_no_diacritics_clean'].duplicated().sum()
print(f"Remaining duplicates: {remaining_duplicates}")

# Drop the temporary cleaning column before saving
train_data_final = train_data_dedup.drop(columns=['poem_text_no_diacritics_clean'])

# Create backup of original file before overwriting
import shutil
import os
from datetime import datetime

backup_file = original_train_file.replace('.csv', f'_backup_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv')
print(f"\nCreating backup: {backup_file}")
shutil.copy2(original_train_file, backup_file)

# Save deduplicated data back to original file
print(f"Saving deduplicated data to: {original_train_file}")
train_data_final.to_csv(original_train_file, index=False)

print(f"\n✅ Successfully removed {removed_count} duplicates from train dataset!")
print(f"✅ Backup saved as: {backup_file}")
print(f"✅ Updated file saved as: {original_train_file}")

# Verify the saved file
print(f"\nVerifying saved file...")
verification_df = pd.read_csv(original_train_file)
print(f"Saved file has {len(verification_df)} rows")
print(f"Original columns preserved: {list(verification_df.columns)}")

=== REMOVING DUPLICATES FROM TRAIN DATASET ===
Initial train dataset size: 2531845 rows
Initial non-empty texts: 2531845

After deduplication:
Final train dataset size: 2305127 rows
Final non-empty texts: 2305127
Removed 226718 duplicate rows
Remaining duplicates: 0

Creating backup: /mnt/data/users/abdelrahman.sadallah/ONEDRIVE/poetry/georoge_decontaminated_train_backup_20251022_071301.csv
Saving deduplicated data to: /mnt/data/users/abdelrahman.sadallah/ONEDRIVE/poetry/georoge_decontaminated_train.csv

✅ Successfully removed 226718 duplicates from train dataset!
✅ Backup saved as: /mnt/data/users/abdelrahman.sadallah/ONEDRIVE/poetry/georoge_decontaminated_train_backup_20251022_071301.csv
✅ Updated file saved as: /mnt/data/users/abdelrahman.sadallah/ONEDRIVE/poetry/georoge_decontaminated_train.csv

Verifying saved file...


/tmp/ipykernel_2916873/98916682.py:61: DtypeWarning: Columns (1,2,6,8,9,10,12,13,17,19,23,24,25,27,28) have mixed types. Specify dtype option on import or set low_memory=False.
  verification_df = pd.read_csv(original_train_file)


Saved file has 2305127 rows
Original columns preserved: ['dataset_name', 'genre', 'location', 'meter', 'overall_explanation', 'poem_id', 'poem_language', 'poem_text', 'poem_title', 'poem_type', 'poem_url', 'poem_verses', 'poet_description', 'poet_english_id', 'poet_era', 'poet_id', 'poet_name', 'poet_url', 'rhyme', 'source', 'verses_explanation', 'poem_text_no_diacritics', 'poet_name_with_diacritics', 'keywords', 'key_phrases', 'corrupted_poem', 'corruption_assigned_template', 'corruption_type', 'corruption_metadata']


In [5]:
from datasets import load_dataset
import re
import pandas as pd

# Load omkarthawakar/FannOrFlop and check duplicates in `poem_verses` after removing diacritics.
# This cell re-uses earlier imports and helpers: pd, clean_text, duplicate_stats are already defined.

def remove_diacritics(text):
    """Remove Arabic diacritics from the given text."""
    arabic_diacritics = re.compile(r'[\u064B-\u0652]')
    return arabic_diacritics.sub('', text)

def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = re.sub(r"[\t\n\r]+", " ", text)
    text = re.sub(r"\s+", " ", text)
    text = text.strip()
    text = re.sub(r"ـ+", "", text)
    return text
def duplicate_stats(df, col='poem_text_no_diacritics_clean'):
    total = len(df)
    non_empty_mask = df[col].str.strip() != ""
    non_empty = non_empty_mask.sum()
    unique_texts = df.loc[non_empty_mask, col].nunique()
    duplicate_rows = df.loc[non_empty_mask, col].duplicated(keep='first').sum()
    duplicate_values = non_empty - unique_texts
    top_dupes = df.loc[non_empty_mask, col].value_counts().head(10)
    return {
        "total_rows": total,
        "non_empty_rows": non_empty,
        "unique_texts": unique_texts,
        "duplicate_rows": duplicate_rows,
        "duplicate_values": duplicate_values,
        "top_duplicates": top_dupes
    }


# Load dataset (pick a split if multiple exist)
ds = load_dataset("omkarthawakar/FannOrFlop")
split_name = "train" if "train" in ds.keys() else list(ds.keys())[0]
df_fann = ds[split_name].to_pandas()

# Create cleaned column for poem_verses
src_col = 'poem_verses'
clean_col = 'poem_verses_no_diacritics_clean'
df_fann[clean_col] = df_fann[src_col].apply(remove_diacritics).apply(clean_text)

# Compute and print duplicate stats using the helper defined earlier
fann_stats = duplicate_stats(df_fann, col=clean_col)

print("=== FannOrFlop duplicates (poem_verses after removing diacritics) ===")
print(f"Total rows: {fann_stats['total_rows']}")
print(f"Non-empty rows: {fann_stats['non_empty_rows']}")
print(f"Unique texts: {fann_stats['unique_texts']}")
print(f"Duplicate rows: {fann_stats['duplicate_rows']}")


=== FannOrFlop duplicates (poem_verses after removing diacritics) ===
Total rows: 6984
Non-empty rows: 6984
Unique texts: 6984
Duplicate rows: 0
